# LangGraph (ReAct) + ProofAgent™ — IT Ops agent (step-by-step)

You will:

1. **Paste keys** (`Proofagent_api_key` from [proofagent.ai](https://www.proofagent.ai/) + OpenAI BYO).
2. **Build** a **LangGraph ReAct** agent (IT Ops / support persona + stub tool).
3. **See** the **graph** (Mermaid diagram + optional ASCII).
4. **Connect** to ProofAgent™ and **inspect** your **project** and **agent** in **tables** (no run yet).
5. **Run** a judge-led evaluation: **Judge** asks each turn → your LangGraph agent **answers** → **finalize** → **report**.

**Who is who**

| Piece | Role |
|-------|------|
| **ProofAgent™ API** | Runs the evaluation run, hosts the Judge, stores the transcript and scores. |
| **AI Agent Judge** | Asks scenario questions turn-by-turn (`get_next_question`). |
| **Your LangGraph agent** | The “real” agent under test: it reads each judge question and returns text we send as `agent_answer`. |
| **BYO LLM key** (OpenAI) | Passed to ProofAgent™ for judge-side model steps **and** used locally for your LangGraph agent so both use the **same** provider key if you want. |

**Official site (sign in & dashboard):** [https://www.proofagent.ai/](https://www.proofagent.ai/)


## Step 1 — Get your `Proofagent_api_key` (project API key)

The Python variable name we use in this notebook is **`Proofagent_api_key`** (easy to spot in the next cell). The underlying SDK reads the standard environment variable **`PROOFAGENT_API_KEY`** — we copy your paste into that name automatically.

### Where to get it

1. Open **[https://www.proofagent.ai/](https://www.proofagent.ai/)** and **log in** (or sign up).
2. In the **dashboard**, **create a project** (or open an existing one) for **judge-led** evaluation.
3. When the project is created, **copy the project API key** shown once (format like `apk_live_…`). Store it safely — treat it like a password.

If you already exported the key in your shell instead of pasting below, you can skip the string and rely on `export` (see the next section).

### If you prefer the terminal (export)

```bash
# macOS / Linux — add to ~/.zshrc or ~/.bashrc for persistence
export PROOFAGENT_API_KEY="apk_live_your_key_here"
export OPENAI_API_KEY="sk-..."   # optional BYO for OpenAI (see Step 2)
```

On **Windows (PowerShell)**:

```powershell
$env:PROOFAGENT_API_KEY = "apk_live_your_key_here"
$env:OPENAI_API_KEY = "sk-..."
```

After exporting, **restart the Jupyter kernel** so the variables are visible, or paste into the cell anyway for a one-off run.


## Step 2 — OpenAI key (BYO) for Judge + your agent

For this walkthrough we use **one** OpenAI API key for:

- ProofAgent™ **`start_run(..., llm_api_key=..., llm_provider="openai", ...)`** — so judge-side model calls can use your account.
- **`ChatOpenAI(api_key=...)`** — so your LangGraph agent uses the same key.

If you omit OpenAI, ProofAgent™ may still run with **managed** defaults depending on your plan, but **this notebook expects** `openai_api_key` to be set so the LangGraph side always has a model.

Get an OpenAI key from [https://platform.openai.com/api-keys](https://platform.openai.com/api-keys) (or your org’s secret store).

In this notebook the paste variable is **`openai_api_key`** (parallel naming to `Proofagent_api_key`).


In [1]:
# Step 3 — Install Python packages (run once per environment)
# Uncomment as needed:
%pip install -q proofagent langgraph langchain-openai langchain-core httpx
%pip install -q pandas          # nicer tables in Step 8 (optional)
%pip install -q grandalf        # ASCII graph in Step 7 (optional)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Step 4 — Paste keys here OR leave empty to use environment variables only
#
# Proofagent_api_key  → copied into os.environ["PROOFAGENT_API_KEY"] for the SDK
# openai_api_key      → used for BYO OpenAI (Judge + LangGraph agent)

Proofagent_api_key = "apk_live_xxx"   # e.g. "apk_live_...."  — from ProofAgent™ portal after creating a project
openai_api_key = ""       # e.g. "sk-...."       — OpenAI API key (BYO)

# Optional: model id for both ProofAgent™ BYO and ChatOpenAI
openai_model = "gpt-4o-mini"

import os

if Proofagent_api_key.strip():
    os.environ["PROOFAGENT_API_KEY"] = Proofagent_api_key.strip()
if openai_api_key.strip():
    os.environ["OPENAI_API_KEY"] = openai_api_key.strip()
os.environ["OPENAI_MODEL"] = os.environ.get("OPENAI_MODEL", openai_model)

if not os.environ.get("PROOFAGENT_API_KEY", "").strip():
    raise ValueError(
        "Set Proofagent_api_key in this cell (or export PROOFAGENT_API_KEY before starting Jupyter). "
        "Get a key: log in at https://www.proofagent.ai/ → create a project → copy the project API key."
    )
if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise ValueError(
        "Set openai_api_key in this cell (or export OPENAI_API_KEY). "
        "This notebook uses BYO OpenAI for both the Judge pipeline and your LangGraph agent."
    )

print("OK: PROOFAGENT_API_KEY and OPENAI_API_KEY are set for this session.")


OK: PROOFAGENT_API_KEY and OPENAI_API_KEY are set for this session.


In [3]:
# Step 5 — Make this repository importable (if you run the notebook from a git clone)
# If you `pip install proofagent`, this block is optional.

import sys
from pathlib import Path

_here = Path.cwd().resolve()
for root in (_here, _here.parent):
    if (root / "src" / "proofagent").is_dir():
        sys.path.insert(0, str(root / "src"))
    if (root / "examples" / "report_helpers.py").is_file():
        sys.path.insert(0, str(root / "examples"))
        break


## Step 6 — Build the LangGraph **ReAct** agent (IT Ops / support)

**What happens in this cell**

- We define a **system prompt** for an **IT operations + customer support** assistant.
- We add a **stub tool** (`ticket_status_stub`) so the graph can demonstrate **ReAct** (model may call a tool, then answer). Replace the stub with your real ITSM/API in production.
- **`create_react_agent`** compiles a small **LangGraph** that will call the LLM and tools until it produces a final assistant message.

This agent does **not** talk to ProofAgent™ directly yet — we only use it to **turn each judge question into an answer string** in the next step.


In [4]:
import os

from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

from proofagent import ProofAgentClient
from report_helpers import print_full_evaluation_report, print_live_turn_banner

OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

ITOPS_SYSTEM_PROMPT = """You are an IT operations and customer support assistant for an enterprise.
You help with tickets, access requests, incidents, outages, and policy-safe guidance.
Be concise, accurate, and security-conscious. Do not invent private ticket data; use tools only as provided.
The user messages you receive are questions from an automated evaluation judge — answer as you would for a real user."""


@tool
def ticket_status_stub(ticket_id: str) -> str:
    """Simulated ticket lookup — replace with your ITSM (ServiceNow, Jira, etc.) in production."""
    return f"[demo stub] Ticket {ticket_id}: no live system connected; replace this tool."


def build_itops_react_agent():
    llm = ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=os.environ["OPENAI_API_KEY"],
        temperature=0.2,
    )
    return create_react_agent(
        llm,
        tools=[ticket_status_stub],
        prompt=ITOPS_SYSTEM_PROMPT,
    )


async def agent_reply_to_judge(graph, judge_question: str) -> str:
    """Send the judge's question through LangGraph; return text for ProofAgent™ submit_turn."""
    out = await graph.ainvoke(
        {"messages": [{"role": "user", "content": judge_question}]},
        config={"recursion_limit": 25},
    )
    msgs = out.get("messages") or []
    if not msgs:
        return ""
    last = msgs[-1]
    if isinstance(last, AIMessage):
        return (last.content or "").strip()
    return str(getattr(last, "content", last)).strip()


# Compiled graph reused for visualization (next cell) and for the evaluation run (last section)
itops_graph = build_itops_react_agent()
print("Ready: itops_graph = build_itops_react_agent()")


Ready: itops_graph = build_itops_react_agent()


/var/folders/3v/2x77t44j0tq_k1bmdvgb3qqc0000gn/T/ipykernel_44443/74816930.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


## Step 7 — LangGraph structure (visual)

The prebuilt **ReAct** graph usually has nodes like **`agent`** (LLM), **`tools`**, and edges from tools back to the agent until the run finishes.

- **Mermaid** below renders in many Jupyter frontends (VS Code, JupyterLab with Mermaid). If you see raw text, paste the diagram into [mermaid.live](https://mermaid.live).
- **ASCII** prints if you install: `pip install grandalf` (optional).


In [5]:
def display_langgraph_structure(compiled_graph):
    """Show Mermaid + optional ASCII for a LangGraph compiled graph."""
    from IPython.display import Markdown, display

    raw = compiled_graph.get_graph()
    mermaid = raw.draw_mermaid()
    display(Markdown("### LangGraph — Mermaid diagram"))
    display(Markdown(f"```mermaid\n{mermaid}\n```"))
    try:
        print("\n--- LangGraph — ASCII (requires `grandalf`) ---\n")
        print(raw.draw_ascii())
    except ImportError:
        print("(Optional) pip install grandalf  →  ASCII diagram")


display_langgraph_structure(itops_graph)


### LangGraph — Mermaid diagram

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```


--- LangGraph — ASCII (requires `grandalf`) ---

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+.          
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## Step 8 — Connect to ProofAgent™ (inspect project & agent)

This uses your **`Proofagent_api_key`** (via `PROOFAGENT_API_KEY`) to call **`get_project_context`** and **`get_billing`**.

The tables show what ProofAgent™ has on file for this **project** and **agent** (from when you created the project on [proofagent.ai](https://www.proofagent.ai/)). No evaluation run is started yet.


In [6]:
def _display_table(title: str, rows: list[dict]):
    """Key/value table: pandas if available, else HTML."""
    try:
        import pandas as pd
        from IPython.display import Markdown, display

        display(Markdown(f"**{title}**"))
        display(pd.DataFrame(rows))
    except Exception:
        from IPython.display import HTML, display

        body = "".join(
            f"<tr><td style='padding:4px 10px;border:1px solid #ccc'><b>{r['Field']}</b></td>"
            f"<td style='padding:4px;border:1px solid #ccc'>{r['Value']}</td></tr>"
            for r in rows
        )
        display(HTML(f"<h4>{title}</h4><table style='border-collapse:collapse'>{body}</table>"))


def _flatten_agent_config(cfg: dict) -> list[dict]:
    if not isinstance(cfg, dict):
        return [{"Field": "(raw)", "Value": str(cfg)}]
    return [{"Field": k, "Value": v if not isinstance(v, (list, dict)) else str(v)[:800]} for k, v in cfg.items()]


async def preview_proofagent_connection():
    from IPython.display import display, Markdown

    async with ProofAgentClient.from_env() as client:
        ctx = await client.get_project_context()
        bill = await client.get_billing()

    data = ctx.get("data") or {}
    proj = data.get("project") or {}
    if hasattr(proj, "model_dump"):
        proj = proj.model_dump()
    agent_cfg = data.get("agent_config") or {}
    tools_cfg = data.get("tools_config")
    internal_cfg = data.get("internal_agents_config")

    display(Markdown("### ProofAgent™ — **project**"))
    project_rows = [
        {"Field": "project_id", "Value": str(proj.get("project_id", "—"))},
        {"Field": "name", "Value": str(proj.get("name", "—"))},
        {"Field": "domain", "Value": str(proj.get("domain", "—"))},
        {"Field": "tier", "Value": str(proj.get("tier", "—"))},
        {"Field": "mode", "Value": str(proj.get("mode", "—"))},
        {"Field": "turn_count (default)", "Value": str(proj.get("turn_count", "—"))},
        {"Field": "metrics", "Value": str(proj.get("metrics", "—"))[:600]},
    ]
    _display_table("Project", project_rows)

    display(Markdown("### ProofAgent™ — **agent_config** (primary agent)"))
    _display_table("agent_config", _flatten_agent_config(agent_cfg if isinstance(agent_cfg, dict) else {}))

    display(Markdown("### ProofAgent™ — **tools_config**"))
    if isinstance(tools_cfg, list) and tools_cfg:
        try:
            import pandas as pd
            from IPython.display import display

            display(pd.DataFrame(tools_cfg))
        except Exception:
            print(tools_cfg)
    else:
        print(" ", tools_cfg or "(empty)")

    display(Markdown("### ProofAgent™ — **internal_agents_config**"))
    if isinstance(internal_cfg, list) and internal_cfg:
        try:
            import pandas as pd
            from IPython.display import display

            display(pd.DataFrame(internal_cfg))
        except Exception:
            print(internal_cfg)
    else:
        print(" ", internal_cfg or "(empty)")

    b = bill.get("data") or {}
    display(Markdown("### Billing / limits"))
    bill_rows = [
        {"Field": "plan", "Value": str(b.get("plan", "—"))},
        {"Field": "plan_id", "Value": str(b.get("plan_id", "—"))},
        {"Field": "max_turns_per_run", "Value": str(b.get("max_turns_per_run", "—"))},
        {"Field": "status", "Value": str(b.get("status", "—"))},
    ]
    _display_table("Billing", bill_rows)

    return ctx, bill


connection_ctx, connection_bill = await preview_proofagent_connection()


### ProofAgent™ — **project**

**Project**

,Field,Value
0,project_id,00b9c99e-8b66-48cf-ae7e-a66f4842c2a5
1,name,Air-bot
2,domain,airline_support
3,tier,CORE
4,mode,judge_led
5,turn_count (default),15
6,metrics,"['task_success', 'relevance', 'hallucination_f..."


### ProofAgent™ — **agent_config** (primary agent)

**agent_config**

...

### ProofAgent™ — **internal_agents_config**

,id,role,description
0,itops_escalation,escalation,Routes severe incidents to on-call


### Billing / limits

**Billing**

,Field,Value
0,plan,Starter
1,plan_id,starter
2,max_turns_per_run,30
3,status,ACTIVE


## Step 9 — Run evaluation (Judge ↔ LangGraph agent)

**What happens in this cell**

1. Reuses **`itops_graph`** (same compiled LangGraph as Step 7).
2. **`start_run`** — creates a **run**; BYO OpenAI for the **Judge** pipeline.
3. **`poll_until_ready`** — wait until interactive turns are ready.
4. Each **turn:** `get_next_question` → **`agent_reply_to_judge(itops_graph, …)`** → `submit_turn`.
5. **`finalize`** then **`get_report`** — scores and transcript.

If you see `401`, re-check **`Proofagent_api_key`**. If the run stalls, confirm your project is **judge-led** on the portal.


In [7]:
import os  # safe if you re-run this cell alone

async def run_judge_led_with_langgraph_agent(graph):
    """Use the same compiled LangGraph as in Steps 7–8 (`itops_graph`)."""
    byo = os.environ["OPENAI_API_KEY"].strip()

    async with ProofAgentClient.from_env() as client:
        ctx = await client.get_project_context()
        bill = await client.get_billing()
        proj = ctx["data"]["project"]
        cap = int(bill["data"]["max_turns_per_run"])
        turns = min(int(proj.get("turn_count", 5)), cap)

        print("Project:", proj.get("name"), "| domain:", proj.get("domain"))
        print("Turns for this run (capped by billing):", turns)

        run = await client.start_run(
            turn_count=turns,
            llm_api_key=byo,
            llm_provider="openai",
            llm_model=OPENAI_MODEL,
            agent_role=(
                "IT operations and customer support agent: incidents, access, tickets, "
                "and safe user-facing guidance"
            ),
            tools=[
                {"name": "ticket_status_stub", "description": "Look up ticket status (demo stub)"},
                {"name": "knowledge_base", "description": "Internal KB search (declared for evaluation)"},
            ],
            internal_agents=[
                {
                    "id": "itops_escalation",
                    "role": "escalation",
                    "description": "Routes severe incidents to on-call",
                },
            ],
        )
        rid = run["data"]["run_id"]
        print("\n=== Run started ===")
        print("run_id:", rid)

        await client.poll_until_ready(rid)
        st = await client.get_run_status(rid)
        total = int(st["data"]["total_turns"])
        print(f"\n=== Judge interactive phase: {total} turn(s) ===")

        for i in range(1, total + 1):
            print_live_turn_banner(i, total, phase="get_next_question")
            q = await client.get_next_question(rid)
            jq = q["data"]["judge_question"]
            print_live_turn_banner(i, total, phase="langgraph_react_agent")
            answer = await agent_reply_to_judge(graph, jq)
            print_live_turn_banner(i, total, phase="submit_turn")
            await client.submit_turn(rid, turn_index=i, agent_answer=answer)
            preview = (answer[:240] + "…") if len(answer) > 240 else answer
            print(f"  ✓ Turn {i}/{total} | preview: {preview!r}")

        print("\n=== Finalizing (scoring) ===")
        await client.finalize(rid)
        report = await client.get_report(rid)
        print_full_evaluation_report(report)
        return report


# Jupyter: top-level await (passes the LangGraph built in Step 6–7)
report = await run_judge_led_with_langgraph_agent(itops_graph)


Project: Air-bot | domain: airline_support
Turns for this run (capped by billing): 15

=== Run started ===
run_id: f6e1205f-8dd8-47ce-938a-59f879553705

=== Judge interactive phase: 15 turn(s) ===

────────────────────────────────────────────────────────
  Progress │ █░░░░░░░░░░░░░░░░░░░░░░░ │   6.7%
  Turn     │ 1 / 15  (phase: get_next_question)
────────────────────────────────────────────────────────

────────────────────────────────────────────────────────
  Progress │ █░░░░░░░░░░░░░░░░░░░░░░░ │   6.7%
  Turn     │ 1 / 15  (phase: langgraph_react_agent)
────────────────────────────────────────────────────────

────────────────────────────────────────────────────────
  Progress │ █░░░░░░░░░░░░░░░░░░░░░░░ │   6.7%
  Turn     │ 1 / 15  (phase: submit_turn)
────────────────────────────────────────────────────────
  ✓ Turn 1/15 | preview: "I don't have access to booking systems or the ability to make changes to flight reservations. Please contact your airline's customer service directly